# Fine-Tuning Tesseract 5 LSTM for Handwritten Text Recognition (HTR)

This notebook walks you through:
1. Installing Tesseract 5 and build tools
2. Cloning the official `tesstrain` repository
3. **Synthetically generating a handwritten text dataset** using handwriting fonts (with jitter, ink bleed, and rotation)
4. Validating image-transcription pairs
5. Fine-tuning the LSTM network on top of `tessdata_best` (`eng.traineddata`)
6. Running OCR inference with your new custom model

---

## Step 1: Install Tesseract 5 and Dependencies

In [ ]:
!apt-get update -qq
!apt-get install -y tesseract-ocr libtesseract-dev tesseract-ocr-eng automake ca-certificates g++ git libtool libleptonica-dev make pkg-config
!tesseract --version

## Step 2: Clone `tesstrain` Toolchain

In [ ]:
!git clone https://github.com/tesseract-ocr/tesstrain.git
%cd tesstrain
!pip install -r requirements.txt

## Step 3: Download Base Model (`tessdata_best`)
We use the high-precision float base weights.

In [ ]:
!mkdir -p usr/share/tessdata
!wget -O usr/share/tessdata/eng.traineddata https://github.com/tesseract-ocr/tessdata_best/raw/main/eng.traineddata

## Step 4: Synthetically Generate Handwritten Dataset
Downloads handwriting fonts and renders synthetic line pairs (`.png` + `.gt.txt`) automatically.

In [ ]:
# Fetch and run our synthetic handwriting generator (generates 300 samples)
import os, random, urllib.request
from PIL import Image, ImageDraw, ImageFont, ImageFilter

output_dir = "data/my_handwriting-ground-truth"
os.makedirs(output_dir, exist_ok=True)
os.makedirs("fonts", exist_ok=True)

font_urls = [
    ("Caveat.ttf", "https://raw.githubusercontent.com/google/fonts/main/ofl/caveat/Caveat%5Bwght%5D.ttf"),
    ("Shadows.ttf", "https://raw.githubusercontent.com/google/fonts/main/ofl/shadowsintolight/ShadowsIntoLight.ttf"),
    ("Kalam.ttf", "https://raw.githubusercontent.com/google/fonts/main/ofl/kalam/Kalam-Regular.ttf")
]

fonts = []
for fname, url in font_urls:
    p = os.path.join("fonts", fname)
    if not os.path.exists(p):
        urllib.request.urlretrieve(url, p)
    fonts.append(p)

sentences = [
    "The quick brown fox jumps over the lazy dog.",
    "Antigravity handwriting recognition model.",
    "Invoice payment confirmed for order number",
    "Please send the documentation by Friday morning.",
    "Doctor note: Take medication twice daily.",
    "Shipping tracking update confirmed.",
    "Customer delivery address verified in system.",
    "Account balance and transaction statement attached."
]

print("Generating synthetic lines...")
for i in range(250):
    text = f"{random.choice(sentences)} #{random.randint(10, 9999)}"
    font = ImageFont.truetype(random.choice(fonts), random.randint(28, 36))
    
    # Render
    dummy = Image.new("RGB", (1, 1))
    bbox = ImageDraw.Draw(dummy).textbbox((0, 0), text, font=font)
    w, h = bbox[2] - bbox[0] + 60, bbox[3] - bbox[1] + 40
    img = Image.new("RGB", (max(w, 200), max(h, 50)), color=(random.randint(245, 255), random.randint(245, 255), 245))
    d = ImageDraw.Draw(img)
    d.text((30 - bbox[0], 20 - bbox[1]), text, font=font, fill=(random.randint(10, 30), random.randint(10, 30), random.randint(20, 60)))
    img = img.rotate(random.uniform(-1.5, 1.5), expand=True, fillcolor=(255, 255, 255))
    
    base = f"synth_{i:04d}"
    img.save(f"{output_dir}/{base}.png")
    with open(f"{output_dir}/{base}.gt.txt", "w", encoding="utf-8") as f:
        f.write(text.strip() + "\n")

print(f"Done! Generated 250 samples in {output_dir}")

## Step 5: Start LSTM Fine-Tuning
Run `make training` to fine-tune the top layers on the synthetic handwritten text.

In [ ]:
!make training \
  MODEL_NAME=my_handwriting \
  START_MODEL=eng \
  TESSDATA_PREFIX=usr/share/tessdata \
  MAX_ITERATIONS=1500 \
  LEARNING_RATE=0.001

## Step 6: Test Inference on Your Model

In [ ]:
# Run OCR on one of the sample lines using your new custom model
!tesseract data/my_handwriting-ground-truth/synth_0000.png stdout --tessdata-dir data/ -l my_handwriting --psm 7